# 03 — Noise Cleaning Performance

Does cleaning actually pay, and by how much?

Two halves, and the order matters:

* **Part A — synthetic.** The population correlation $C$ is known, so the oracle
  $\xi_i = u_i^\top C u_i$ is computable and we can check the shrinkage map
  *pointwise*, not just aggregate loss. Every bug found here is just a bug.
  Every bug found on real data is confounded with data problems.
* **Part B — rolling backtest.** Walk-forward, strictly out of sample.

Headline metric is **not Sharpe**. It is realised out-of-sample volatility and the
risk ratio $\mathcal R = \sigma_{\text{realised}}/\sigma_{\text{predicted}}$, which
should approach 1. The sample covariance says the portfolio is safer than it is;
that calibration failure is what cleaning fixes, and it shows up with far less
noise than any return-based metric.

Reads `results/tables/baseline.csv` from notebook 01 rather than refitting.

In [ ]:
import sys, pathlib, importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the repository root by walking up for the `src` package.  Matching on
# a hardcoded folder name is what broke before -- the check said "notebooks"
# while the directory was "notebook", so it silently put the notebook folder
# itself on sys.path and `import src` failed.  The folder has since been
# renamed, but walking up does not care what it is called, and works from the
# repo root, from this folder, or from anywhere below either.
def _find_root():
    start = globals().get("__vsc_ipynb_file__") or pathlib.Path.cwd()
    here = pathlib.Path(start).resolve()
    for cand in [here, *here.parents]:
        if (cand / "src" / "__init__.py").exists():
            return cand
    raise RuntimeError(
        f"no src/__init__.py found at or above {here}; open this notebook from "
        "inside the cloned repository")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# The Jupyter kernel is frequently NOT the interpreter a terminal `pip install`
# reaches.  Name it explicitly, so a missing package points at the environment
# to fix instead of looking like the package was never installed.
_missing = [m for m in ("numpy", "pandas", "scipy", "matplotlib")
            if importlib.util.find_spec(m) is None]
if _missing:
    raise ImportError(
        f"this kernel is missing: {', '.join(_missing)}\n"
        f"  kernel interpreter : {sys.executable}\n"
        f"  install into THAT interpreter:\n"
        f'      "{sys.executable}" -m pip install -r "{ROOT / "requirements.txt"}"\n'
        f"  then restart the kernel (a plain `pip install` in a terminal may\n"
        f"  have targeted a different environment entirely)")

from src.data import (prepare, factor_correlation, simulate_returns,
                      dispersed_spectrum)
from src.spectral import spectrum, fit_mp_bulk, mp_edges
from src.estimators import build, REGISTRY, invert_spike, spike_overlap
from src.backtest import run_backtest, summarise, min_variance

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (7, 4),
                     "axes.grid": True, "grid.alpha": .25, "font.size": 9})
FIGDIR = ROOT / "results" / "figures"; FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = ROOT / "results" / "tables";  TABDIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(7)

METHODS = ["sample", "linear", "clipping", "factor", "rie", "nonlinear"]
COLORS  = dict(zip(METHODS + ["oracle"], ["C7", "C0", "C2", "C4", "C3", "C1", "k"]))

base = pd.read_csv(TABDIR / "baseline.csv").iloc[0] if (TABDIR / "baseline.csv").exists() else None
if base is not None:
    print(f"baseline from notebook 01: q_eff={base['q_eff']:.3f}  "
          f"sigma^2={base['sigma2']:.3f}  lambda_+={base['lambda_plus']:.3f}")

## Part A — Synthetic ground truth

### A.1 The shrinkage map

Every estimator here is rotationally invariant: it keeps the sample eigenvectors
and transforms only the eigenvalues, so the whole design space is the map
$\xi(\lambda)$. Plotting all of them against the oracle on one axis is the single
most informative figure in the project.

In [ ]:
def make_case(N, T, kind="factor", k=3, spikes=(20., 8., 3.), rng=None):
    rng = RNG if rng is None else rng
    if kind == "factor":
        C = factor_correlation(N, k=k, loading_sd=.5, rng=rng)
    elif kind == "spiked":
        V = np.linalg.qr(rng.standard_normal((N, len(spikes))))[0]
        C = np.eye(N) + V @ np.diag(np.array(spikes) - 1.) @ V.T
        d = np.sqrt(np.diag(C)); C = C / np.outer(d, d)
    elif kind == "dispersed":
        pop = dispersed_spectrum(N)
        Q = np.linalg.qr(rng.standard_normal((N, N)))[0]
        C = Q @ np.diag(pop) @ Q.T
        d = np.sqrt(np.diag(C)); C = C / np.outer(d, d)
    else:
        raise ValueError(kind)
    X = simulate_returns(C, T, rng=rng)
    X = (X - X.mean(0)) / X.std(0, ddof=1)
    return C, X

N, T = 400, 800
C, X = make_case(N, T, kind="spiked")
fits = {m: build(m).fit(X) for m in METHODS}
fits["oracle"] = build("oracle", C_true=C).fit(X)
lam = fits["sample"].lambda_
f = fit_mp_bulk(lam, q0=N / T)
print(f"q = {N/T:.3f}   q_eff = {f['q_eff']:.3f}   sigma^2 = {f['sigma2']:.3f}   "
      f"lambda_+ = {f['lambda_plus']:.3f}   spikes = {f['n_exclude']}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
hi = f["lambda_plus"] * 1.05

for ax, (lo_x, hi_x, ttl) in zip(axes, [(0, hi, "bulk"), (0, lam.max() * 1.05, "full range")]):
    m = (lam >= lo_x) & (lam <= hi_x)
    ax.plot([lo_x, hi_x], [lo_x, hi_x], color="C7", lw=1, ls="--", label="sample (identity)")
    for name in ["linear", "clipping", "rie", "nonlinear"]:
        ax.plot(lam[m], fits[name].eigenvalues_[m], ".", ms=4,
                color=COLORS[name], label=name)
    ax.plot(lam[m], fits["oracle"].eigenvalues_[m], "k+", ms=5, label="oracle")
    ax.axvline(f["lambda_plus"], color="k", ls=":", lw=1)
    ax.set_xlabel("sample eigenvalue $\\lambda$"); ax.set_title(ttl)
    if ttl == "full range":
        ax.set_xscale("log"); ax.set_yscale("log")
axes[0].set_ylabel("cleaned eigenvalue $\\xi$")
axes[0].legend(fontsize=7.5, loc="upper left")
fig.suptitle(f"Shrinkage maps, N={N}, T={T}, q={N/T:.2f}", y=1.01)
fig.tight_layout(); fig.savefig(FIGDIR / "03_shrinkage_maps.png", bbox_inches="tight")
plt.show()

### A.2 Correctness test: RIE vs analytical nonlinear shrinkage

These are two independent numerical routes to the *same* asymptotic limit — a
complex resolvent with a finite imaginary shift $\eta$ on one side, a kernel
density and Hilbert transform on the other. **If they disagree in the bulk, one
implementation is wrong.** This is the cheapest correctness check available and
it costs nothing to run on every case.

In [ ]:
rows = []
for kind in ["factor", "spiked", "dispersed"]:
    for (Nc, Tc) in [(200, 1000), (300, 600), (400, 500)]:
        Cc, Xc = make_case(Nc, Tc, kind=kind)
        sp = spectrum(Xc)
        e = {m: build(m).fit(Xc, spectrum_=sp) for m in ["rie", "nonlinear"]}
        orc = build("oracle", C_true=Cc).fit(Xc, spectrum_=sp)
        fc = fit_mp_bulk(sp[0], q0=Nc / Tc)
        bulk = sp[0] <= fc["lambda_plus"]
        rel = np.abs(e["rie"].eigenvalues_[bulk] - e["nonlinear"].eigenvalues_[bulk])
        rel = np.median(rel / e["nonlinear"].eigenvalues_[bulk])
        # Oracle tracking is measured on the bulk. The spikes dominate the full
        # norm and are handled by a separate branch, so including them would
        # measure the spike inversion rather than the shrinkage map.
        gap = lambda xi: (np.linalg.norm(xi[bulk] - orc.eigenvalues_[bulk])
                          / np.linalg.norm(sp[0][bulk] - orc.eigenvalues_[bulk]))
        rows.append(dict(case=kind, N=Nc, T=Tc, q=Nc / Tc,
                         bulk_disagreement=rel,
                         rie_gap=gap(e["rie"].eigenvalues_),
                         nonlin_gap=gap(e["nonlinear"].eigenvalues_)))

xcheck = pd.DataFrame(rows)
print(xcheck.to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
d_max, g_max = xcheck["bulk_disagreement"].max(), xcheck["rie_gap"].max()
print(f"\nmax bulk disagreement, RIE vs nonlinear: {d_max:.4f}"
      f"   [{'PASS' if d_max < 0.05 else 'FAIL'} at 5%]")
print(f"max residual gap to oracle (bulk), RIE:   {g_max:.4f}"
      f"   [{'PASS' if g_max < 0.35 else 'FAIL'} at 0.35]")
print("gap = ||xi - oracle|| / ||lambda - oracle||; 0 = oracle, 1 = no better than sample.")

### A.3 Where the gain lives: loss against $q$

At small $q$ the true oracle map is nearly affine, so linear shrinkage is nearly
optimal and there is almost nothing for the nonlinear methods to win. The gain
appears only when $q$ is substantial. This sweep is why the project targets
$q \in [0.3, 0.8]$ — outside it you would prove nothing either way.

In [ ]:
qs, loss = [], {m: [] for m in METHODS + ["oracle"]}
Nq = 300
for Tq in [3000, 1500, 1000, 750, 600, 500, 430, 375]:
    Cq, Xq = make_case(Nq, Tq, kind="factor")
    sp = spectrum(Xq)
    qs.append(Nq / Tq)
    for m in METHODS:
        e = build(m).fit(Xq, spectrum_=sp)
        loss[m].append(np.linalg.norm(e.sigma_ - Cq, "fro"))
    loss["oracle"].append(
        np.linalg.norm(build("oracle", C_true=Cq).fit(Xq, spectrum_=sp).sigma_ - Cq, "fro"))

fig, ax = plt.subplots(figsize=(7, 4.2))
for m in METHODS + ["oracle"]:
    ax.plot(qs, loss[m], "o-", ms=4, color=COLORS[m],
            lw=2.2 if m == "oracle" else 1.4,
            ls="--" if m == "oracle" else "-", label=m)
ax.axvspan(0.3, 0.8, color="k", alpha=.06)
ax.set_xlabel("q = N/T"); ax.set_ylabel("$\\|\\hat\\Sigma - C\\|_F$")
ax.set_title("Loss against q (shaded: the regime where cleaning is worth measuring)")
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(FIGDIR / "03_loss_vs_q.png", bbox_inches="tight"); plt.show()

print(pd.DataFrame(loss, index=np.round(qs, 3)).rename_axis("q")
      .to_string(float_format=lambda v: f"{v:8.2f}"))

### A.4 What cannot be recovered

The spike inversion gives the BBP detectability threshold for free. A population
spike is recoverable only if

$$\theta > \sigma^2\left(1 + \sqrt q\right)$$

Below it the eigenvector overlap $\omega$ hits zero: the sample "spike" carries no
directional information at all, and no estimator in this class — or any other —
can recover it. Worth plotting, because it is the cleanest statement of the
method's hard limit.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for qv, c in [(0.2, "C0"), (0.5, "C2"), (0.8, "C3")]:
    th = np.linspace(1.001, 8, 400)
    ax.plot(th, spike_overlap(th, qv), color=c, lw=1.8, label=f"q = {qv}")
    ax.axvline(1 + np.sqrt(qv), color=c, ls=":", lw=1)
ax.set_xlabel("population spike $\\theta$"); ax.set_ylabel("eigenvector overlap $\\omega$")
ax.set_title("BBP detectability: $\\omega = 0$ below $\\theta = \\sigma^2(1+\\sqrt{q})$")
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(FIGDIR / "03_bbp_threshold.png", bbox_inches="tight"); plt.show()

## Part B — Rolling backtest

Walk-forward, no overlap between the estimation window and the holdout it is
evaluated on. Weights come from the unconstrained global minimum variance
portfolio $w \propto \Sigma^{-1}\mathbf 1$: it isolates the covariance estimate
and, because it inverts $\Sigma$, it is maximally sensitive to the smallest
eigenvalues — exactly where cleaning bites.

Estimators are fitted on devolatilised, standardised returns, so `sigma_` is a
*correlation* matrix. The driver reassembles
$\Sigma = D^{1/2}\hat{\mathcal C}D^{1/2}$ from the window's marginal volatilities
before building weights; without that step the predicted volatility is
dimensionless and the risk ratio is meaningless.

In [ ]:
Nb, Tb = 250, 2000
Cb = factor_correlation(Nb, k=4, loading_sd=.45, rng=RNG)
Rb = pd.DataFrame(simulate_returns(Cb, Tb, rng=RNG) * 0.011,
                  index=pd.bdate_range("2016-01-04", periods=Tb),
                  columns=[f"S{i:03d}" for i in range(Nb)])
vol = pd.Series(np.exp(np.cumsum(RNG.normal(0, .025, Tb))), index=Rb.index)
Rb = Rb.mul(vol, axis=0)

panel = prepare(Rb, halflife=63)
print("panel", panel.shape, " q =", round(panel.q, 3))

LOOKBACK, HOLDOUT = 500, 21
specs = {m: (m, {}) for m in METHODS}
res = run_backtest(panel, specs, lookback=LOOKBACK, holdout=HOLDOUT)
print(f"\nq per window = {Nb/LOOKBACK:.2f}")

In [ ]:
tab = summarise(res)
tab["realised_vol_ann"] = tab["realised_vol"]
print(tab[["realised_vol", "risk_ratio", "vol_reduction_vs_sample",
           "mean_turnover", "sharpe"]]
      .to_string(float_format=lambda v: f"{v:10.4f}"))

print("\nrisk_ratio = realised / predicted. 1.0 is perfect calibration;")
print("above 1 means the estimator claims the portfolio is safer than it is.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

order = tab.index.tolist()
axes[0].bar(order, tab["realised_vol"], color=[COLORS[m] for m in order])
axes[0].set_ylabel("annualised realised vol"); axes[0].set_title("Out-of-sample risk")
axes[0].tick_params(axis="x", rotation=45)

rr = (res.realised / res.predicted)
# `labels=` was deprecated in matplotlib 3.9 and removed in 3.11; setting the
# tick labels directly works on every version.
axes[1].boxplot([rr[m].dropna() for m in order], showfliers=False)
axes[1].set_xticks(range(1, len(order) + 1))
axes[1].set_xticklabels(order)
axes[1].axhline(1.0, color="k", ls="--", lw=1)
axes[1].set_ylabel("realised / predicted"); axes[1].set_title("Risk ratio per rebalance")
axes[1].tick_params(axis="x", rotation=45)

axes[2].bar(order, tab["mean_turnover"], color=[COLORS[m] for m in order])
axes[2].set_ylabel("mean one-way turnover"); axes[2].set_title("Weight stability")
axes[2].tick_params(axis="x", rotation=45)

fig.tight_layout(); fig.savefig(FIGDIR / "03_backtest_summary.png", bbox_inches="tight")
plt.show()

In [ ]:
# Risk ratio through time: does calibration hold, or only on average?
fig, ax = plt.subplots(figsize=(8, 3.8))
for m in ["sample", "linear", "rie", "nonlinear"]:
    ax.plot(rr.index, rr[m].rolling(6, min_periods=1).median(),
            color=COLORS[m], lw=1.5, label=m)
ax.axhline(1.0, color="k", ls="--", lw=1)
ax.set_ylabel("risk ratio (6-rebalance median)"); ax.set_xlabel("rebalance date")
ax.set_title("Calibration through time"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGDIR / "03_risk_ratio_time.png", bbox_inches="tight")
plt.show()

print(res.diagnostics.describe().loc[["mean", "std", "min", "max"]]
      .to_string(float_format=lambda v: f"{v:8.4f}"))

### B.1 Sensitivity to $\eta$

$\eta$ is the RIE's only tuning knob. Too small and the poles of the empirical
resolvent leak through; too large and spikes bleed into the bulk. The default
$\eta \sim N^{-1/2}$ sits above the typical eigenvalue spacing $O(1/N)$ but below
the scale on which $\rho$ varies.

**This plot is required, not optional** — it is the honest way to show the result
is not an artifact of one tuning choice.

In [ ]:
etas = [0.02, 0.05, 0.08, 0.12, 0.2, 0.3]
rows = []
for e_ in etas:
    r_ = run_backtest(panel, {"rie": ("rie", {"eta": e_})},
                      lookback=LOOKBACK, holdout=HOLDOUT, verbose=False)
    t_ = summarise(r_)
    rows.append(dict(eta=e_, realised_vol=t_.loc["rie", "realised_vol"],
                     risk_ratio=t_.loc["rie", "risk_ratio"],
                     turnover=t_.loc["rie", "mean_turnover"]))
sens = pd.DataFrame(rows).set_index("eta")
default_eta_val = Nb ** -0.5

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(sens.index, sens["realised_vol"], "o-", color="C3", label="realised vol")
ax.axvline(default_eta_val, color="k", ls=":", lw=1.2,
           label=f"default $N^{{-1/2}}$ = {default_eta_val:.3f}")
ax.axhline(tab.loc["sample", "realised_vol"], color="C7", ls="--", lw=1, label="sample")
ax.set_xlabel("$\\eta$"); ax.set_ylabel("annualised realised vol")
ax.set_title("RIE sensitivity to the resolvent regularisation"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGDIR / "03_eta_sensitivity.png", bbox_inches="tight")
plt.show()

print(sens.to_string(float_format=lambda v: f"{v:9.4f}"))

## Conclusions

In [ ]:
tab.to_csv(TABDIR / "cleaning_performance.csv")
sens.to_csv(TABDIR / "eta_sensitivity.csv")
xcheck.to_csv(TABDIR / "rie_vs_nonlinear.csv", index=False)

best = tab.index[0]
lines = [
    "",
    "CONCLUSION",
    "----------",
    f"Rolling GMV, N={Nb}, lookback={LOOKBACK} (q={Nb/LOOKBACK:.2f}), "
    f"{HOLDOUT}-day holdouts, {len(rr)} rebalances.",
    "",
    f"Best estimator by out-of-sample volatility: {best}, "
    f"{100*tab.loc[best,'vol_reduction_vs_sample']:.1f}% below the sample covariance.",
    f"Risk ratio: sample {tab.loc['sample','risk_ratio']:.2f} -> {best} "
    f"{tab.loc[best,'risk_ratio']:.2f} (1.00 is perfect calibration).",
    f"Turnover: sample {tab.loc['sample','mean_turnover']:.3f} -> {best} "
    f"{tab.loc[best,'mean_turnover']:.3f}.",
    "",
    f"RIE and analytical nonlinear shrinkage agree in the bulk to "
    f"{100*xcheck['bulk_disagreement'].max():.2f}% at worst across nine "
    "synthetic cases, which is the correctness check on both implementations.",
    "",
    "Caveats worth stating in the writeup, not hiding:",
    " - these are synthetic returns; the real-data version of this table is the",
    "   actual deliverable and the ordering may not survive intact",
    " - Sharpe is reported but should not be read as evidence: it is dominated by",
    "   return noise, and the covariance claim is about risk, not return",
    " - the unconstrained GMV inverts Sigma and so flatters cleaning. A long-only",
    "   variant (backtest.min_variance_long_only, needs cvxpy) is the harder test",
    "   and the advantage surviving it would be the stronger result",
]
print(chr(10).join(lines))